# Train IBAnet: spectra to probabilistic elemental depth profiles

IBAnet follows `docs/draft.tex`: one residual 1D CNN branch per detector and a unidirectional GRU
that decodes layers from surface to depth. Shared heads produce conditional mean **Y**, conditional standard deviation **R**, and presence
probability **P** for every layer/element. Training minimizes the Bernoulli plus
masked Gaussian negative log-likelihood. Targets are elemental areal densities,
not normalized concentrations.

Install `pip install -e ".[training,plot]"`. Like `train_lrn_multilayer.ipynb`,
this notebook loads generated `layers_XX` datasets, trains in stages, evaluates
held-out samples, and exports an iBeamLab package. Its inputs and outputs run in
the opposite direction. All configured detector spectra are required.

The Gaussian in the draft is untruncated and can draw negative densities.
IBAnet constrains its conditional mean positive, but retains that Gaussian
likelihood. Native deployment exports the nonnegative posterior mean **P × Y**;
uncertainties and exact absence probabilities remain available in PyTorch.
The mean need not contain exact zero padding, and is not a predicted layer count.

In [ ]:
from pathlib import Path
import sys
import re
import os
import math
import json
import copy
try:
    import tomllib
except ImportError:
    import tomli as tomllib

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "examples":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
EXAMPLES = ROOT / "examples"
import ibeamlab as ibl
from ibeamlab import IBAnet, IBAnetLoss
from ibeamlab._native import native

## 1. Dataset and measurement contract

Choose one dataset collection generated with the same fixed experimental setup
and reference methods. Mixing calibrations, exposures, geometries, or pileup
conditions does not produce a spectra-only inverse model with a well-defined
reference setup. The loader checks stored setup, sample templates, method
references, and parameter targets against this configuration. Supply only
compatible datasets; channel padding below extends the same reference grid.

In [ ]:
DATASET_ROOT = EXAMPLES / "datasets"
CONFIG_PATH = EXAMPLES / "multilayer-generation.toml"
MAX_LAYERS = 10
SEED = 7
OUTPUT_DIR = EXAMPLES / "artifacts" / "ibanet_multilayer_compressed16"

configuration = ibl.load_generation_configuration(CONFIG_PATH)
if configuration.variations:
    raise ValueError("IBAnet requires fixed beam/detector settings; choose a config without setup variations")
if not 1 <= MAX_LAYERS <= configuration.maximum_layers:
    raise ValueError("MAX_LAYERS exceeds the configuration")
study = configuration.study(MAX_LAYERS)
ELEMENTS = tuple(configuration.elements)
LABELS = tuple(d.label for d in study.experiment.detectors)
LAYER_DATASETS = {}
for path in sorted(DATASET_ROOT.rglob("layers_*")):
    match = re.fullmatch(r"layers_(\d+)", path.name)
    if path.is_dir() and match and 1 <= int(match.group(1)) <= MAX_LAYERS:
        LAYER_DATASETS.setdefault(int(match.group(1)), []).append(path)
if not LAYER_DATASETS:
    raise FileNotFoundError(f"No layers_XX datasets under {DATASET_ROOT}")
print("Detector order:", LABELS)
print("Element order:", ELEMENTS)
print("Maximum layers:", MAX_LAYERS)

## 2. Load spectra and construct areal-density targets

Each target entry is `thickness[layer] * concentration[layer, element]` in
`1e15 atoms/cm2`. Deeper layers absent from a dataset are exact zero rows.
Concentrations are validated, not silently renormalized. Raw spectra are
concatenated in detector order; CNN branches keep detector boundaries separate.
This eager loader holds the selected datasets in memory, as in the LRN notebook.

In [ ]:
def template_table(text):
    value = tomllib.loads(text)
    value.pop("format", None)
    value.pop("format_version", None)
    return value

def load_case(dataset_path, layer_count):
    dataset = ibl.open_dataset(dataset_path)
    if not dataset.metadata.complete or not len(dataset):
        raise ValueError(f"Incomplete or empty dataset: {dataset_path}")
    case = configuration.study(layer_count)
    config = case._config()
    provenance = tomllib.loads(dataset._reader.metadata.generation_config_toml)
    expected_setup = template_table(native.sample.setup_to_toml(config.setup))
    expected_sample = template_table(native.sample.sample_to_toml(config.sample))
    if provenance.get("setup") != expected_setup or provenance.get("sample") != expected_sample:
        raise ValueError(f"Dataset template/setup differs from CONFIG_PATH: {dataset_path}")
    expected_methods = [{"label": m.label, "iba_method": m.iba_method,
                         "reference_file": Path(m.reference_file).as_posix()} for m in config.methods]
    if provenance.get("methods") != expected_methods:
        raise ValueError(f"Dataset method/reference differs from CONFIG_PATH: {dataset_path}")
    names = tuple(dataset.metadata.parameter_names)
    if len(set(names)) != len(names) or set(names) != set(case.parameter_names):
        raise ValueError(f"Parameter names do not match the case study: {dataset_path}")
    stored_targets = {p["name"]: p["target"] for p in provenance.get("parameters", [])}
    for p in case.parameters:
        expected_target = ({"type": "layer_thickness", "layer": p.layer} if p.kind == "layer_thickness"
                           else {"type": "species_concentration", "layer": p.layer, "element": p.element})
        if stored_targets.get(p.name) != expected_target:
            raise ValueError(f"Parameter target differs for {p.name}: {dataset_path}")
    rows = np.asarray(dataset.parameters, dtype=np.float32)
    columns = {name: i for i, name in enumerate(names)}
    densities = np.zeros((len(rows), MAX_LAYERS, len(ELEMENTS)), np.float32)
    for layer_index in range(layer_count):
        layer = case.experiment.sample.layers[layer_index]
        thickness = np.full(len(rows), layer.thickness, np.float32)
        fractions = np.tile([next(s.concentration for s in layer.species if s.element == element)
                             for element in ELEMENTS], (len(rows), 1)).astype(np.float32)
        for p in case.parameters:
            if p.layer != layer_index:
                continue
            if p.kind == "layer_thickness":
                thickness = rows[:, columns[p.name]]
            elif p.kind == "concentration":
                fractions[:, ELEMENTS.index(p.element)] = rows[:, columns[p.name]]
        if np.any(thickness < 0) or np.any(fractions < 0) or not np.allclose(fractions.sum(1), 1, atol=1e-6, rtol=0):
            raise ValueError(f"Invalid thickness/composition: {dataset_path}")
        densities[:, layer_index] = thickness[:, None] * fractions
    spectra = {label: np.asarray(dataset.spectra(label), dtype=np.float32) for label in LABELS}
    if any(not np.isfinite(x).all() or np.any(x < 0) for x in spectra.values()) or not np.isfinite(densities).all():
        raise ValueError(f"Nonfinite or negative training data: {dataset_path}")
    return spectra, densities

loaded = []
SPECTRUM_LENGTHS = {label: 0 for label in LABELS}
for layer_count, paths in sorted(LAYER_DATASETS.items()):
    for path in paths:
        spectra, densities = load_case(path, layer_count)
        for label in LABELS:
            SPECTRUM_LENGTHS[label] = max(SPECTRUM_LENGTHS[label], spectra[label].shape[1])
        loaded.append((spectra, densities))
        print(f"Loaded {path}: {len(densities):,} samples, {layer_count} layers")
x_parts, a_parts = [], []
for spectra, densities in loaded:
    x_parts.append(np.concatenate([
        np.pad(spectra[label], ((0, 0), (0, SPECTRUM_LENGTHS[label] - spectra[label].shape[1])))
        for label in LABELS], axis=1))
    a_parts.append(densities)
x_raw = np.concatenate(x_parts)
a_raw = np.concatenate(a_parts)
del loaded, x_parts, a_parts
print("Raw spectra:", x_raw.shape, "EDP:", a_raw.shape)

## 3. Split, scale, and build the CNN/GRU

Validation and test each receive 10%, capped at 20,000 and 10,000 samples.
Fit channel normalization only on training `log1p(expected counts)`. The normalization
buffers are inside the exported ONNX graph, so deployment takes raw counts.
`TARGET_SCALE` rescales both the Gaussian mean and standard deviation.
A sample NLL sums all layer/element entries; target scaling changes its numerical
value. Use the same scale for training and evaluation.

The encoder uses GroupNorm; context and decoder inputs use LayerNorm.
The GRU receives spectrum context and a learned position embedding at every
layer. Its hidden state carries information toward deeper layers; no previous
targets or sampled outputs are supplied. There is one Gaussian slab per cell.
`HEAD_HIDDEN_SIZES` now configures shared heads after the GRU. Old CNN/MLP
PyTorch checkpoints require retraining; existing exported packages still work.
Keep the target scale identical when comparing old and new NLL values.

With `POISSON_NOISE=True`, stored spectra are interpreted as noise-free
expected counts. Validation and test each receive a fixed seeded Poisson draw
in this section. Section 4 draws fresh raw training counts for every batch,
before the model applies `log1p` and standardization. Targets are unchanged.
Set this option to `False` if stored spectra already include counting noise,
and rerun section 3 when changing it. Normalization remains fitted on clean
training expected counts, so it does not depend on a particular noise draw.

The split is by sample. If several datasets contain repeated simulations or
related scans, use a grouped split before interpreting held-out performance.

A 1x1 convolution compresses the final CNN channels to `COMPRESSION_CHANNELS`
(default 16) before flattening, preserving energy positions and reducing the
context projection by 8x with the default 128-channel encoder. The compressed
architecture needs new training; the default output directory is separate from
previous runs. Use `None` to disable compression.


In [ ]:
TARGET_SCALE = 1e-5
POISSON_NOISE = True  # Stored spectra must be noise-free expected counts
CNN_CHANNELS = (32, 64, 128, 128)
COMPRESSION_CHANNELS = 16  # 1x1 channel projection before flattening
HEAD_HIDDEN_SIZES = (128,)
DECODER_HIDDEN_SIZE = 256
DECODER_LAYERS = 2
LAYER_EMBEDDING_SIZE = 32
DROPOUT = 0.1
KERNEL_SIZE = 7
MINIMUM_STD = 1e-5  # in scaled target units

if len(x_raw) < 10:
    raise ValueError("At least 10 samples are needed")
order = np.random.default_rng(SEED).permutation(len(x_raw))
n_val = min(max(1, int(0.1 * len(order))), 20_000)
n_test = min(max(1, int(0.1 * len(order))), 10_000)
train_indices = order[:len(order) - n_val - n_test]
val_indices = order[len(order) - n_val - n_test:len(order) - n_test]
test_indices = order[-n_test:]
log_train = np.log1p(x_raw[train_indices])
input_mean = log_train.mean(axis=0, dtype=np.float64).astype(np.float32)
input_std = np.maximum(log_train.std(axis=0, dtype=np.float64), 1e-3).astype(np.float32)
del log_train
if not np.isfinite(TARGET_SCALE) or TARGET_SCALE <= 0:
    raise ValueError("TARGET_SCALE must be positive")
x_train, x_val, x_test = (x_raw[i] for i in (train_indices, val_indices, test_indices))
a_train, a_val, a_test = (a_raw[i] * TARGET_SCALE for i in (train_indices, val_indices, test_indices))

# Fixed raw-count draws keep validation/checkpoint comparisons and test/native
# parity reproducible. Training counts remain clean until each batch is drawn.
def fixed_poisson_counts(expected_counts, seed):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    noisy = np.empty_like(expected_counts)
    for start in range(0, len(expected_counts), 512):
        stop = min(start + 512, len(expected_counts))
        noisy[start:stop] = torch.poisson(
            torch.from_numpy(expected_counts[start:stop]), generator=generator).numpy()
    return noisy

if POISSON_NOISE:
    x_val = fixed_poisson_counts(x_val, SEED + 1)
    x_test = fixed_poisson_counts(x_test, SEED + 2)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = IBAnet(study, SPECTRUM_LENGTHS, elements=ELEMENTS,
               cnn_channels=CNN_CHANNELS, kernel_size=KERNEL_SIZE,
               compression_channels=COMPRESSION_CHANNELS,
               head_hidden_sizes=HEAD_HIDDEN_SIZES, minimum_std=MINIMUM_STD,
               decoder_hidden_size=DECODER_HIDDEN_SIZE, decoder_layers=DECODER_LAYERS,
               layer_embedding_size=LAYER_EMBEDDING_SIZE, dropout=DROPOUT,
               input_mean=input_mean, input_std=input_std).to(device)
loss_function = IBAnetLoss()
print("Device:", device, "Train/validation/test:", len(x_train), len(x_val), len(x_test))
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## 4. Train and restore the best validation checkpoint

The staged AdamW schedule and gradient clipping follow the forward notebook.
Use zero DataLoader workers by default for Windows/Jupyter compatibility.
Each improved validation result is saved immediately. Checkpoint metadata,
normalization buffers, split indices, and configuration are saved together.
`LOAD_CHECKPOINT=True` by default: rerunning this cell after interruption
automatically loads compatible best weights from the same output directory.
The optimizer is fresh and the configured schedule restarts; this is fine-tuning,
not exact resumption. Architecture, configuration, split, and normalization
checks still reject incompatible checkpoints. For fresh training, rebuild the
model in section 3 and choose a new `OUTPUT_DIR`; existing weights are preserved.
If the saved checkpoint used a different noise policy (including older
checkpoints without noise metadata), compatible weights are reused, historical
losses are cleared, and the best-loss baseline is recomputed on the current
fixed validation spectra. Rerunning section 4 does not redraw validation/test
noise; section 3 recreates the same draws from their seeds.
NLL can be negative, so loss plots use a linear scale.

The loss omits the constant `0.5 * log(2*pi)` per present cell. This changes
reported NLL values but leaves training gradients unchanged. Presence loss is
`-M*log(P) - (1-M)*log(1-P)`, evaluated stably from logits.


In [ ]:
BATCH_SIZE = 512
VALIDATION_BATCH_SIZE = 2048
DATALOADER_WORKERS = 0
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
TRAINING_SCHEDULE = [
    {"learning_rate": 3e-4, "epochs": 20},
    {"learning_rate": 1e-4, "epochs": 30},
    {"learning_rate": 5e-5, "epochs": 20},
    {"learning_rate": 1e-5, "epochs": 10},
]
LOAD_CHECKPOINT = True  # Automatically reuse a compatible best checkpoint on rerun
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
weights_path = OUTPUT_DIR / "ibanet_state.pt"
PIN_MEMORY = device.type == "cuda"
training_noise_generator = torch.Generator(device=device).manual_seed(SEED + 3)
loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(a_train)),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=DATALOADER_WORKERS,
                    pin_memory=PIN_MEMORY, generator=torch.Generator().manual_seed(SEED))

def evaluate_loss(inputs, targets):
    model.eval()
    total = 0.0
    with torch.inference_mode():
        for start in range(0, len(inputs), VALIDATION_BATCH_SIZE):
            x = torch.from_numpy(inputs[start:start + VALIDATION_BATCH_SIZE]).to(device)
            a = torch.from_numpy(targets[start:start + VALIDATION_BATCH_SIZE]).to(device)
            total += float(loss_function(model(x), a)) * len(x)
    return total / len(inputs)

signature = dict(max_layers=MAX_LAYERS, elements=list(ELEMENTS), labels=list(LABELS),
                 spectrum_lengths=SPECTRUM_LENGTHS, target_scale=TARGET_SCALE,
                 architecture=model.architecture,
                 input_noise=dict(poisson=POISSON_NOISE, validation_seed=SEED + 1, test_seed=SEED + 2),
                 config_text=CONFIG_PATH.read_text(),
                 datasets=[str(p.resolve()) for paths in LAYER_DATASETS.values() for p in paths])
history = []
best_loss = float("inf")
best_state = None
if weights_path.exists():
    if not LOAD_CHECKPOINT:
        raise FileExistsError(f"Checkpoint already exists: {weights_path}. Set LOAD_CHECKPOINT=True to continue, or choose a new OUTPUT_DIR for fresh training.")
    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
    # Noise-policy changes can reuse structurally compatible weights, but old
    # losses must not be compared against the new fixed validation realization.
    stored_signature = dict(checkpoint["signature"])
    stored_noise = stored_signature.pop("input_noise", None)
    current_signature = {k: v for k, v in signature.items() if k != "input_noise"}
    if stored_signature != current_signature:
        raise ValueError("Checkpoint schema/configuration differs")
    noise_changed = stored_noise != signature["input_noise"]
    if any(not torch.equal(checkpoint[name], torch.from_numpy(indices))
           for name, indices in (("train_indices", train_indices), ("val_indices", val_indices),
                                 ("test_indices", test_indices))):
        raise ValueError("Checkpoint split differs")
    np.testing.assert_allclose(checkpoint["model_state_dict"]["input_mean"].numpy(), input_mean)
    np.testing.assert_allclose(checkpoint["model_state_dict"]["input_std"].numpy(), input_std)
    model.load_state_dict(checkpoint["model_state_dict"])
    history = [] if noise_changed else list(checkpoint["history"])
    if noise_changed:
        print("Input noise policy changed; resetting loss history and evaluating saved weights on the current validation spectra")
    best_loss = evaluate_loss(x_val, a_val)
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"Loaded best checkpoint: {weights_path} (validation NLL {best_loss:.6g}); restarting the configured schedule with a fresh optimizer")
optimizer = torch.optim.AdamW(model.parameters(), lr=TRAINING_SCHEDULE[0]["learning_rate"],
                              weight_decay=WEIGHT_DECAY)

def save_best():
    checkpoint = {"model_state_dict": best_state, "signature": signature,
                  "history": history, "best_validation_nll": best_loss,
                  "train_indices": torch.from_numpy(train_indices),
                  "val_indices": torch.from_numpy(val_indices),
                  "test_indices": torch.from_numpy(test_indices)}
    temporary = weights_path.with_suffix(".pt.tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(weights_path)

try:
    with tqdm(total=sum(s["epochs"] for s in TRAINING_SCHEDULE), desc="IBAnet epochs") as progress:
        for stage in TRAINING_SCHEDULE:
            for group in optimizer.param_groups:
                group["lr"] = stage["learning_rate"]
            for _ in range(stage["epochs"]):
                model.train()
                total, seen = 0.0, 0
                for inputs, targets in loader:
                    inputs, targets = inputs.to(device), targets.to(device)
                    if POISSON_NOISE:
                        # Raw expected counts -> fresh counting noise -> model's
                        # embedded log1p and training-fitted standardization.
                        inputs = torch.poisson(inputs, generator=training_noise_generator)
                    optimizer.zero_grad(set_to_none=True)
                    loss = loss_function(model(inputs), targets)
                    if not torch.isfinite(loss):
                        raise FloatingPointError("Nonfinite NLL; inspect targets and scaling")
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True)
                    optimizer.step()
                    total += float(loss.detach()) * len(inputs)
                    seen += len(inputs)
                validation = evaluate_loss(x_val, a_val)
                if not math.isfinite(validation):
                    raise FloatingPointError("Nonfinite validation NLL")
                history.append((total / seen, validation))
                if validation < best_loss:
                    best_loss = validation
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    save_best()
                progress.update(1)
                progress.set_postfix(train=total / seen, validation=validation)
except KeyboardInterrupt:
    print("Interrupted; restoring the best completed validation checkpoint")
if best_state is None:
    raise RuntimeError("No completed validation checkpoint; train at least one epoch")
model.load_state_dict(best_state)
model.eval()
save_best()
print("Best validation NLL:", best_loss)
print("Checkpoint:", weights_path)

## 5. Held-out EDP and uncertainty evaluation

Report posterior-mean error in physical units, presence Brier score, and test
NLL. Conditional ±1.96R coverage is evaluated only where the true element is
present; it is a Gaussian-slab diagnostic, not a 95% interval for the full
spike-and-slab posterior. The conditional-independence model does not enforce
correlated compositions or a contiguous layer mask.

In [ ]:
model.eval()  # Disable dropout even when this cell is run out of order
outputs = []
with torch.inference_mode():
    for start in range(0, len(x_test), BATCH_SIZE):
        output = model(torch.from_numpy(x_test[start:start + BATCH_SIZE]).to(device))
        outputs.append((output.Y.cpu().numpy(), output.R.cpu().numpy(), output.P.cpu().numpy()))
y = np.concatenate([o[0] for o in outputs]) / TARGET_SCALE
r = np.concatenate([o[1] for o in outputs]) / TARGET_SCALE
p = np.concatenate([o[2] for o in outputs])
truth = a_test / TARGET_SCALE
estimate = p * y
present = truth > 0
print("Test NLL (scaled units):", evaluate_loss(x_test, a_test))
print("Posterior-mean MAE (1e15 atoms/cm2):", float(np.abs(estimate - truth).mean()))
print("Presence Brier score:", float(np.square(p - present).mean()))
print("Conditional-slab 95% coverage:", float((np.abs(truth[present] - y[present]) <= 1.96 * r[present]).mean())
      if present.any() else "no present elements")
fig, axes = plt.subplots(min(4, len(truth)), 3, figsize=(15, 3 * min(4, len(truth))), squeeze=False,
                         constrained_layout=True)
for i, row in enumerate(axes):
    common_max = max(float(truth[i].max()), float(estimate[i].max()), 1e-6)
    for axis, matrix, title in zip(row, (truth[i], estimate[i], p[i]),
                                   ("True EDP", "Posterior mean EDP", "Presence probability")):
        im = axis.imshow(matrix, aspect="auto", vmin=0, vmax=1 if title == "Presence probability" else common_max)
        axis.set_xticks(range(len(ELEMENTS)), ELEMENTS, rotation=45)
        axis.set_ylabel("Layer (surface to depth)")
        axis.set_title(f"Sample {i}: {title}")
        fig.colorbar(im, ax=axis)
plt.show()
losses = np.asarray(history)
plt.plot(losses[:, 0], label="train")
plt.plot(losses[:, 1], label="validation")
plt.xlabel("Epoch")
plt.ylabel("NLL per sample (scaled target units)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Export and verify native inference

The version 4 package contains the reference sample/setup, detector channel
lengths, ordered elements, EDP dimensions and units. Its ONNX graph includes
log1p plus fitted normalization and outputs flattened **P × Y**. Package output
scaling restores physical units. It exports no hard existence threshold, so
small positive densities can remain in absent elements/layers.

Use the saved PyTorch checkpoint for conditional **Y, R** and sampling. A raw Gaussian
draw may be negative or have noncontiguous layers; project/filter it explicitly
before using it as a physical optimizer initialization. Such projection changes
the distribution from the draft. This notebook does not silently apply it.

The current datasets have pileup disabled. Export writes
`need_pileup_subtraction = true` in inverse metadata: external inference callers
can supply spectra and detector settings through `InverseInput` for automatic correction.
Already pileup-free spectra need no further subtraction.


The parity check uses `predict_prepared()` because dataset spectra already match
training conditions. For experiments, call `predict(InverseInput(...))`: the
library removes pileup, rebins, and scales exposure before the embedded transforms.
Set `pileup_already_removed=True` for spectra already corrected for pileup.

Native predictions also expose `presence_probability` and `posterior_std`, with the same layer/element shape as `values`. Standard deviation is in physical areal-density units and includes presence uncertainty.


In [ ]:
package_path = OUTPUT_DIR / f"ibanet_{MAX_LAYERS}layers.zip"
export_suffix = 1
while package_path.exists():
    # A previous export may exist even if its parity check was interrupted.
    package_path = OUTPUT_DIR / f"ibanet_{MAX_LAYERS}layers_{export_suffix:03d}.zip"
    export_suffix += 1
# Verify against the same current CPU/eval weights used for ONNX export.
# `estimate` from section 5 may be stale or computed with CUDA kernels.
export_model = copy.deepcopy(model).cpu().eval()
check_count = min(8, len(x_test))
check_inputs = np.array(x_test[:check_count], dtype=np.float32, copy=True)
with torch.inference_mode():
    export_reference = torch.cat([
        export_model(torch.from_numpy(row[None])).mean
        for row in check_inputs], dim=0).numpy() / TARGET_SCALE
export_model.export(package_path, output_inverse_factor=TARGET_SCALE, need_pileup_subtraction=True,
             pileup_fudge_factor_seconds=0.4e-6)
metadata = native.model.ModelPackage.open(package_path).metadata
assert metadata.format_version == 4
assert metadata.class_name == "IBAnet"
assert metadata.inverse.need_pileup_subtraction
np.testing.assert_allclose(metadata.inverse.pileup_fudge_factor_seconds, 0.4e-6)
assert list(metadata.inverse.output_edp.elements) == list(ELEMENTS)
assert metadata.inverse.output_edp.max_layers == MAX_LAYERS
assert metadata.input_transform.type == "identity"
np.testing.assert_allclose(metadata.output_transform.factor, TARGET_SCALE)

native_model = ibl.InverseModel(package_path)
measurements = []
for row in check_inputs:
    offset = 0
    item = {}
    for label in LABELS:
        width = SPECTRUM_LENGTHS[label]
        item[label] = ibl.Spectrum(label, row[offset:offset + width])
        offset += width
    measurements.append(item)
native_edp = np.stack([result.values for result in native_model.predict_prepared(measurements)])
max_export_difference = float(np.abs(native_edp - export_reference).max())
print(f"Native vs fresh CPU PyTorch maximum absolute difference: {max_export_difference:.6g}")
np.testing.assert_allclose(native_edp, export_reference, rtol=2e-4, atol=1e-3,
                           err_msg="Native export differs from current CPU/eval PyTorch predictions")
print("Verified native/PyTorch posterior-mean parity:", package_path)